# 02. Basic Parsing and Guardrails

**Purpose:** Core prompt engineering notebook: baseline prompting, prompt tuning, and guardrails (Task 1 + guardrails part of Task 2).

**Student Experience (3 Acts):**
- **Act I — Baseline:** Run the starter prompt and record a baseline score.
- **Act II — Power of Prompts:** Run a deliberately bad prompt, then improve it to beat the baseline.
- **Act III — Make it Robust:** Toggle `USE_GUARDRAILS` to handle invalid / non-address inputs safely.

**Key Takeaway:** Prompt quality and system robustness matter more than “just using a better model.”


## Step 0: Set up environment & authenticate

In [3]:
!git clone https://github.com/cedrickopp-esmt/esmt-workshop-deutsche-bank.git

import os, sys, time
from pathlib import Path
PROJECT_ROOT = [
    it for it in [Path("src"), Path('esmt-workshop-deutsche-bank/src')]
    if (it / 'esmt_workshop').exists()
][0].parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
%pip install -q -r {PROJECT_ROOT}/requirements.txt

import pandas as pd
from esmt_workshop.preset_params import get_preset_params
from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report, publish_to_leaderboard
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses
from esmt_workshop.authenticate import authenticate

STUDENT_EMAIL = "cedric.kopp@esmt.berlin"
preset_params = get_preset_params()

fatal: destination path 'esmt-workshop-deutsche-bank' already exists and is not an empty directory.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.1/719.1 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 5.0 MB/s eta 0:00:00


## Step 1: Small-Run Prompt Lab (edit **one cell only**)


**Goal:** Try different prompts and observe how decoding parameters affect the model’s behavior.

**How to use the next cell:**

1. **Bad prompt run (to observe poor performance):**  
   - Set `PROMPT_TO_RUN = BAD_PROMPT_TEMPLATE`  
   - Set `STAGE_NAME = 'prompt_tuned'`  
   - Run the cell → observe degraded performance.

2. **Your best prompt (prompt tuning):**  
   - Set `PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE`  
   - Set `STAGE_NAME = 'prompt_tuned'`  
   - Run the cell → try to beat the baseline.

3. Toggle `USE_GUARDRAILS = False/True` and re-run → observe how guardrails affect robustness.

4. `TEMPERATURE`, `TOP_P`, `TOP_K`


**Rule:** edit:
- `PROMPT_TO_RUN`
- `STAGE_NAME` (as instructed above)
- `USE_GUARDRAILS`


**Suggested experiments (do these in order):**
1. Run once with **`TEMPERATURE = 0.0`** (most consistent).
2. Increase temperature (e.g., **0.2 → 0.7**) and re-run. Watch for more variability and occasional formatting drift.
3. Keep temperature fixed and vary **`TOP_P`** (e.g., 0.8 vs 1.0) to see how nucleus sampling changes outputs.
4. Keep temperature/top_p fixed and vary **`TOP_K`** (e.g., 20 vs 80) to see how restricting candidate tokens affects stability.

**Note:** For extraction tasks, lower randomness is usually better—but you should *observe* the trade-off yourself.


Leave everything else unchanged.

In [ ]:
# -----------------------------
# PROMPT (edit here)
# -----------------------------
# Baseline prompt (run this first)

# Bad prompt example (copy/paste to see poor performance)
BAD_PROMPT_TEMPLATE = "Parse the address according to {schema} and return JSON. Address: {address}"

# Your best prompt (edit this to beat the baseline)
STUDENT_PROMPT_TEMPLATE = '''Put your text here {address} {schema}
'''

# Choose which prompt to run:
#   - BAD_PROMPT_TEMPLATE       (use with STAGE_NAME='prompt_tuned')
#   - STUDENT_PROMPT_TEMPLATE   (use with STAGE_NAME='prompt_tuned')
PROMPT_TO_RUN = BAD_PROMPT_TEMPLATE


# -----------------------------
# PARAMS (from original notebooks)
# -----------------------------
NOTEBOOK_NAME = '02_basic_parsing_and_guardrails'

# IMPORTANT:
# - Use STAGE_NAME='baseline' when running BASELINE_PROMPT_TEMPLATE
# - Use STAGE_NAME='prompt_tuned' when running BAD/STUDENT prompts
STAGE_NAME = 'baseline'  # must be one of: baseline, prompt_tuned, advanced, two_stage

MODEL_NAME = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
COUNTRY_MODEL = os.getenv('WORKSHOP_COUNTRY_MODEL', '')  # optional

# Generation params (keep consistent while tuning prompts)
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = 40

# Execution params (don't change)
MAX_TOKENS = preset_params["MAX_TOKENS"]
MAX_WORKERS = preset_params["MAX_WORKERS"]

# Guardrails toggle (students should switch this True/False)
USE_GUARDRAILS = False


# -----------------------------
# DATA (small run)
# -----------------------------
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')
dev_small = dev_df.head(10).copy()


# -----------------------------
# EXECUTE PIPELINE
# -----------------------------
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_small,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

# Generate the report (required line)
report = evaluate_predictions(pred_df, dev_small)

# -----------------------------
# DISPLAY RESULTS (do not edit)
# -----------------------------
print(report['summary'])
display(report['field_metrics'])

In [ ]:
report['mismatches']

In [ ]:
report['merged']

In [ ]:
report['usage_metadata']

In [ ]:
from esmt_workshop.evaluation import calculate_cost
calculate_cost(report['usage_metadata'])

## Step 2: Full Dataset Run (your **publishable** score)

Now run your **best prompt** on the **entire dev dataset**.  
This **micro_accuracy** is the score you will use later when publishing.

**Before running this cell:**
- Set `PROMPT_TO_RUN = STUDENT_PROMPT_TEMPLATE`
- Make sure `STAGE_NAME = 'prompt_tuned'`
- Decide whether to use guardrails: set `USE_GUARDRAILS = True/False`

All other parameters are already defined above and should not be changed here.

In [ ]:
# Do not edit this file
t0 = time.perf_counter()

pred_df = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,  # should be 'prompt_tuned'
    model=MODEL_NAME,
    country_model=COUNTRY_MODEL if COUNTRY_MODEL else None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=USE_GUARDRAILS,
    custom_prompt_template=PROMPT_TO_RUN,
    max_workers=MAX_WORKERS,
)

runtime_sec = time.perf_counter() - t0
print('Runtime (sec):', round(runtime_sec, 3))

report = evaluate_predictions(pred_df, dev_df)

# -----------------------------
# DISPLAY RESULTS (do not edit)
# -----------------------------
print(report['summary'])
display(report['field_metrics'])

In [ ]:
report['mismatches']

In [ ]:
report['merged']

In [ ]:
report['usage_metadata']

In [ ]:
from esmt_workshop.evaluation import calculate_cost
calculate_cost(report['usage_metadata'])

## Step 3: Publish to Leaderboard (Placeholder)

This workshop may use a leaderboard for scoring. The publishing mechanism depends on the organizer's submission endpoint.
Keep this as a placeholder for now.


In [ ]:
publish_to_leaderboard(report, STUDENT_EMAIL)